# A Complete Experiment
In [building_blocks.ipynb] we demonstrated how to use each of the basic building blocks that the framework uses. However, this is impractical to use regularly, and introduces the chance of mistakes. Instead, you should run models through `Experiment` which will be demonstrated below.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
# We can generate a config manually
from lwfm.core.config import ExperimentConfig

raw_config = {
    "name": "test_experiment",
    "seed": 1,
    "device": "cpu",
    "model": {"name": "resnet18", "num_classes": 10},
    "data": {},
    "losses": [{}],
    "metrics": [{}],
    "training": {},
    "recipe": {
        "stages": [
            {"name": "fit", "type": "fit"},
            {"name": "eval_val", "type": "evaluate", "split": "val"},
        ]
    },
    "outputs": {},
    "deployment": {},
}
config = ExperimentConfig.model_validate(raw_config)

print(config)

name='test_experiment' seed=1 device='cpu' model={'name': 'resnet18', 'num_classes': 10} data={} losses=[{}] metrics=[{}] training={} recipe={'stages': [{'name': 'fit', 'type': 'fit'}, {'name': 'eval_val', 'type': 'evaluate', 'split': 'val'}]} outputs={} deployment={}


In [3]:
# Or load automatically
from lwfm.core.yaml import load_yaml_config

yaml = "configs/experiments/smoke.yaml"
yaml_path = Path.cwd().parent / yaml

config = load_yaml_config(yaml_path)

print(config)

name='smoke_resnet18' seed=42 device='cpu' model={'node': 'student', 'inputs': 'inputs.image', 'outputs': 'logits', 'type': 'torchvision', 'name': 'resnet18', 'num_classes': 10, 'pretrained': False, 'preprocess': {'resize': [64, 64], 'normalization': {'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225]}}} data={'type': 'fake_classification', 'num_classes': 10, 'num_train': 32, 'num_val': 16, 'image_size': [64, 64], 'batch_size': 8, 'num_workers': 0} losses=[{'name': 'ce', 'type': 'cross_entropy', 'inputs': ['student.logits', 'targets.class'], 'weight': 1.0}] metrics=[{'name': 'top1', 'type': 'topk_accuracy', 'inputs': ['student.logits', 'targets.class'], 'kwargs': {'top_k': 1, 'num_classes': 10, 'average': 'micro'}}, {'name': 'top5', 'type': 'topk_accuracy', 'inputs': ['student.logits', 'targets.class'], 'kwargs': {'top_k': 5, 'num_classes': 10, 'average': 'micro'}}] training={'optimizer': {'name': 'sgd', 'lr': 0.01}, 'max_epochs': 1, 'max_steps': 2, 'scheduler': {'name': 'cosi

In [4]:
# Now we can
from pprint import pprint

from lwfm.orchestration.experiment import Experiment

experiment = Experiment(config)
pprint(experiment.context)

RunContext(experiment_name='smoke_resnet18',
           config=ExperimentConfig(name='smoke_resnet18',
                                   seed=42,
                                   device='cpu',
                                   model={'inputs': 'inputs.image',
                                          'name': 'resnet18',
                                          'node': 'student',
                                          'num_classes': 10,
                                          'outputs': 'logits',
                                          'preprocess': {'normalization': {'mean': [0.485,
                                                                                    0.456,
                                                                                    0.406],
                                                                           'std': [0.229,
                                                                                   0.224,
                                   

In [12]:
# This is great, but the interesting part is actually RUNNING an experiment.
# We call using the with so we safely close the experiment
with Experiment(config) as experiment:
    experiment.run()